# 5 Reward - 奖励目标

经过 SFT 阶段之后，模型在一定程度上应该能够按照我们的要求来生成一些古典诗词，但效果可能仍然有瑕疵。除了使用高质量的训练数据进行进一步 SFT 训练之外，也可以使用另外的后训练手段——强化学习（RL）。

这里我们会给强化学习阶段做一些必要的准备工作，主要是围绕着奖励目标来做，通过有明确的奖励目标，就可以引导模型向着更加稳定的方向进行输出。我们准备制作 3 个奖励目标：

1. **押韵检查**：对诗词进行押韵的检查（基于规则）
2. **格式检查**：对诗词格式进行检查（基于规则）
3. **作者风格匹配**：判断诗词与作者风格是否匹配（基于神经网络分类器）

本 Notebook 的重点是第 3 个奖励目标，我们将训练一个二元分类器来判断"作者-诗词"配对是否合理。

**分类器的关键改进**：
- 使用双向 Transformer 层（移除 causal mask），让作者信息也能"看到"诗词内容
- 使用全局平均池化（而不是只用最后一个 token），充分利用所有位置的信息
- 冻结 GPT 权重，只训练双向层和分类头，训练高效

In [1]:
import random

import torch

from nanopoet.common import CharTokenizer
from nanopoet.dataset import load_raw_data, split_data
from nanopoet.model import GPTLanguageModel

# 初始化随机种子，让重复执行的结果稳定
random.seed(12345)
torch.manual_seed(12345)

# 首先加载数据、设备信息
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
data = load_raw_data("../raw")

# 初始化分词器
tokenizer = CharTokenizer("".join(["".join(list(d.values())) for d in data]))


In [2]:
from nanopoet.common import filter_by_author, update_poem_author

# 数据集 只取在目标作者范围内的数据
filtered_data = [d for d in data if filter_by_author(d)]
updated_data = [update_poem_author(d) for d in filtered_data]

# 初始化模型结构
block_size = 256
model = GPTLanguageModel(
    vocab_size=tokenizer.vocab_size,
    emb_size=256,
    block_size=block_size,
    layer_num=8,
    head_num=8,
    dropout=0.1,
).to(device)

# 加载 SFT Train 的训练结果
pretrain_state = torch.load("./output/03_mid_train_model.pt", map_location=device)
model.load_state_dict(pretrain_state, strict=True)

<All keys matched successfully>

In [3]:
# ==================== 奖励目标 1: 押韵检查 ====================
from pypinyin import pinyin, Style
from collections import Counter


def check_rhyme(content) -> float:
    """
    押韵奖励函数
    
    要求：每个句号结尾的最后一个字都需要押韵
    返回：0-1 的奖励值，1 表示完美押韵
    """
    # 用句号将诗词切分，取每一句最后一个字符
    sentences = [s for s in content.split("。") if s.strip()]
    if len(sentences) < 2:
        # 句子太少，无法检查押韵
        return 0.0

    # 提取每句最后一个字的韵母
    end_chars = [s[-1] for s in sentences]
    end_rhymes = [pinyin(c, style=Style.FINALS)[0][0] for c in end_chars]

    # 统计最常见的韵母出现次数
    rhyme_counter = Counter(end_rhymes)
    most_common_rhyme = max(list(rhyme_counter.values()))

    # 如果有 70% 的句子都押同一个韵，则认为符合要求
    target_rhyme_number = int(len(end_chars) * 0.7)
    return min(float(most_common_rhyme) / float(target_rhyme_number), 1.0)


# 测试押韵检查
test_poem = updated_data[2]["content"]
rhyme_score = check_rhyme(test_poem)
print(f"测试诗词: {test_poem}")
print(f"押韵分数: {rhyme_score:.2f}")

测试诗词: 唐虞亦人耳，四海可高謝。哀哉斗升故，諂妄兩憑架。心明物自賓，能整故能暇。會當棄人事，面壁度九夏。
押韵分数: 1.00


In [4]:
# ==================== 奖励目标 2: 格式检查 ====================

def generate_format_code(content):
    """
    对诗词进行编码，返回每一句的字数信息
    
    例如：七言律诗 -> "7-7-7-7-7-7-7-7"
    """
    tmp = content.split("。")
    lines = []
    for c in tmp:
        lines.extend(c.split("，"))
    lines = [l for l in lines if l.strip()]
    return "-".join([str(len(l)) for l in lines])


def extract_format(data):
    """
    根据数据集统计各个形式诗词的格式规则
    
    返回：{style: [format1, format2, ...]} 的字典
    """
    style_formats = {}
    for poem in data:
        style = poem["style"]
        format_code = generate_format_code(poem["content"])
        if style not in style_formats:
            style_formats[style] = []
        if format_code not in style_formats[style]:
            style_formats[style].append(format_code)
    return style_formats


def check_format(style, content, format_data) -> int:
    """
    格式检查函数（严格模式）
    
    返回：1 表示格式完全符合，0 表示不符合
    """
    style_formats = format_data.get(style)
    if style_formats is None:
        return 0

    format_code = generate_format_code(content)
    return 1 if format_code in style_formats else 0


# 提取格式数据
format_data = extract_format(filtered_data)
print(f"共提取 {len(format_data)} 种诗词格式")

# 查看一些典型格式
for style in ["七言律诗", "五言绝句", "沁园春"]:
    if style in format_data:
        formats = format_data[style]
        print(f"{style}: {formats[:3]}")  # 只显示前3种格式

# 测试格式检查
test_style = updated_data[2]["style"]
test_content = updated_data[2]["content"]
format_score = check_format(test_style, test_content, format_data)
print(f"\n测试: {test_style} - {generate_format_code(test_content)}")
print(f"格式检查: {'通过' if format_score == 1 else '未通过'}")

共提取 145 种诗词格式
七言律诗: ['7-7-7-7-7-7-7-7']
五言绝句: ['5-5-5-5']
沁园春: ['4-4-4-5-4-4-4-4-4-7-3-5-4-6-8-5-4-4-4-4-4-7-3-5-4', '4-4-4-5-4-4-4-4-4-7-3-5-4-2-4-8-5-4-4-4-4-4-7-3-5-4']

测试: 五言律诗 - 5-5-5-5-5-5-5-5
格式检查: 通过


In [5]:
from nanopoet.model import FeedForward
import torch
import torch.nn as nn
from torch.nn import functional as F


# ==================== 奖励目标 3: 作者风格匹配分类器 ====================
# 我们在基于 SFT 模型训练的基础上进行一些修改，设计一个二元判别器，然后合成一些数据样本对它进行训练，让它能够对诗词内容是否符合诗人风格进行评分，输出一个0-1的数字。

class BidirectionalMultiHeadAttention(nn.Module):
    """双向多头注意力（无 Causal Mask）"""

    def __init__(self, emb_size, head_num, dropout=0.0):
        super().__init__()
        assert emb_size % head_num == 0

        self.emb_size = emb_size
        self.head_num = head_num
        self.head_size = emb_size // head_num

        self.qkv = nn.Linear(emb_size, 3 * emb_size, bias=False)
        self.proj = nn.Linear(emb_size, emb_size)
        self.dropout = nn.Dropout(dropout)

        # 关键：不注册 causal mask（与 GPT 的唯一区别）

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.emb_size, dim=-1)

        q = q.view(B, T, self.head_num, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.head_num, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.head_num, self.head_size).transpose(1, 2)

        wei = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
        # 不做 causal masking，全连接注意力
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        out = self.dropout(out)
        return out


class BidirectionalTransformerBlock(nn.Module):
    """双向 Transformer Block"""

    def __init__(self, emb_size, head_num, dropout=0.0):
        super().__init__()
        self.sa = BidirectionalMultiHeadAttention(emb_size, head_num, dropout)
        self.ffwd = FeedForward(emb_size, dropout)
        self.ln1 = nn.LayerNorm(emb_size)
        self.ln2 = nn.LayerNorm(emb_size)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BinaryClassifier(nn.Module):
    """
    二元分类器
    
    架构：
    1. GPT 编码器（causal，复用预训练权重）
    2. 双向 Transformer 层（全局感知，让作者信息也能看到诗词内容）
    3. 全局平均池化（利用所有位置的信息）
    4. 分类头
    """

    def __init__(self, gpt_model: GPTLanguageModel, freeze_base=True, num_bidirectional_layers=2):
        super().__init__()
        self.gpt = gpt_model

        # 是否冻结 GPT 参数（只训练新增的双向层和分类头）
        if freeze_base:
            for param in self.gpt.parameters():
                param.requires_grad = False

        # 双向 Transformer 层（新增，随机初始化）
        self.bidirectional_blocks = nn.Sequential(*[
            BidirectionalTransformerBlock(
                emb_size=self.gpt.emb_size,
                head_num=self.gpt.head_num,
                dropout=0.1
            )
            for _ in range(num_bidirectional_layers)
        ])

        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(self.gpt.emb_size, self.gpt.emb_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.gpt.emb_size // 2, 1)
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # 阶段1：通过 GPT 编码（causal，利用预训练知识）
        tok_emb = self.gpt.token_embedding(idx)
        pos_emb = self.gpt.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.gpt.blocks(x)
        x = self.gpt.ln_f(x)

        # 阶段2：通过双向层（全局感知）
        x = self.bidirectional_blocks(x)

        # 阶段3：全局平均池化
        pooled = x.mean(dim=1)  # [B, T, D] -> [B, D]

        # 阶段4：分类
        logits = self.classifier(pooled).squeeze(-1)  # [B]

        # 计算损失
        loss = None
        if targets is not None:
            loss = F.binary_cross_entropy_with_logits(logits, targets.float())

        return logits if loss is None else (logits, loss)

In [6]:
# 初始化模型并加载预训练权重
# 创建分类器（冻结 GPT，只训练双向层和分类头）
classifier = BinaryClassifier(
    gpt_model=model,
    freeze_base=True,  # 冻结 GPT 权重
    num_bidirectional_layers=2  # 2 层双向 Transformer
).to(device)

# 统计参数
gpt_params = sum(p.numel() for p in classifier.gpt.parameters() if p.requires_grad)
bidirectional_params = sum(p.numel() for p in classifier.bidirectional_blocks.parameters())
classifier_head_params = sum(p.numel() for p in classifier.classifier.parameters())
total_trainable = gpt_params + bidirectional_params + classifier_head_params

print(f"\n参数统计：")
print(f"  GPT 参数: {gpt_params:,} (已冻结)")
print(f"  双向层参数: {bidirectional_params:,}")
print(f"  分类头参数: {classifier_head_params:,}")
print(f"  总可训练参数: {total_trainable:,}")


参数统计：
  GPT 参数: 0 (已冻结)
  双向层参数: 1,577,984
  分类头参数: 33,025
  总可训练参数: 1,611,009


In [7]:
# 准备分类器训练数据
# 数据准备策略：
# - 正样本：保留原始作者 + 诗词内容
# - 负样本：随机替换作者 + 诗词内容
# - 剔除 style 和 title 字段，只保留 author 和 content
# - 使用 encode_poem 函数进行编码

from nanopoet.common import encode_poem

# 划分训练集和验证集
train_data, val_data = split_data(updated_data)
print(f"训练集: {len(train_data)} 首")
print(f"验证集: {len(val_data)} 首")

# 提取所有作者列表
all_authors = list(set([p["author"] for p in train_data]))
print(f"作者数量: {len(all_authors)}")


def create_positive_sample(poem):
    """
    创建正样本：保留原始作者
    剔除 style 和 title，只保留 author 和 content
    """
    sample_dict = {
        "author": poem["author"],
        "content": poem["content"]
    }
    return encode_poem(sample_dict)


def create_negative_sample(poem, all_authors):
    """
    创建负样本：随机替换作者
    剔除 style 和 title，只保留 author 和 content
    """
    # 随机选择一个不同的作者
    wrong_authors = [a for a in all_authors if a != poem['author']]
    wrong_author = random.choice(wrong_authors)
    
    sample_dict = {
        "author": wrong_author,
        "content": poem["content"]
    }
    return encode_poem(sample_dict)


# 生成训练样本（正负样本各一半）
def generate_samples(data, all_authors, num_samples):
    samples = []
    labels = []
    
    for i in range(num_samples):
        poem = random.choice(data)
        
        if i % 2 == 0:
            # 正样本
            sample = create_positive_sample(poem)
            label = 1
        else:
            # 负样本
            sample = create_negative_sample(poem, all_authors)
            label = 0
        
        samples.append(sample)
        labels.append(label)
    
    return samples, labels


# 生成训练和验证样本
train_samples, train_labels = generate_samples(train_data, all_authors, 1000)
val_samples, val_labels = generate_samples(val_data, all_authors, 200)

# 编码成 token ids
train_encoded = [torch.tensor(tokenizer.encode(s), dtype=torch.long) for s in train_samples]
val_encoded = [torch.tensor(tokenizer.encode(s), dtype=torch.long) for s in val_samples]

print(f"\n样本示例：")
print(f"  正样本: {train_samples[0][:60]}...")
print(f"  负样本: {train_samples[1][:60]}...")
print(f"  标签: {train_labels[:4]}")
print(f"\n编码后长度统计：")
print(f"  训练集平均长度: {sum(len(x) for x in train_encoded) / len(train_encoded):.1f}")
print(f"  验证集平均长度: {sum(len(x) for x in val_encoded) / len(val_encoded):.1f}")

训练集: 19423 首
验证集: 2159 首
作者数量: 15

样本示例：
  正样本: BA陆游aC農家耕作苦，雨暘每關念。種黍蹋麴糵，終歲勤收斂。社甕雖草草，酒味亦醇釅。長歌南陌頭，百年應不厭。c...
  负样本: BA王维aC睡味清酣鼻息輕，碧幮文簟喜涼生。片雲過處失簾影，急雨來時聞瓦聲。索虜尚憑三輔險，散關未下九天兵。白頭漫倚詩豪...
  标签: [1, 0, 1, 0]

编码后长度统计：
  训练集平均长度: 55.0
  验证集平均长度: 54.3


In [8]:
# 训练分类器

def get_classifier_batch(encoded_samples, labels, batch_size, block_size, device):
    """获取一个batch的分类数据"""
    indices = torch.randint(len(encoded_samples), (batch_size,))
    batch_samples = [encoded_samples[i] for i in indices]
    batch_labels = [labels[i] for i in indices]

    # 截断和填充
    batch_x = []
    for sample in batch_samples:
        if len(sample) > block_size:
            sample = sample[:block_size]
        # 简单处理：不填充，直接使用变长序列（需要后续改进）
        batch_x.append(sample)

    # 填充到统一长度
    max_len = max(len(x) for x in batch_x)
    padded_x = []
    for x in batch_x:
        if len(x) < max_len:
            pad_len = max_len - len(x)
            x = torch.cat([x, torch.zeros(pad_len, dtype=torch.long)])
        padded_x.append(x)

    batch_x = torch.stack(padded_x).to(device)
    batch_y = torch.tensor(batch_labels, dtype=torch.long).to(device)

    return batch_x, batch_y


# 训练配置
batch_size = 16
learning_rate = 1e-4  # 较小的学习率
total_steps = 100  # Notebook 演示，只训练少量步骤
eval_interval = 20
eval_iters = 5

# 优化器（只优化可训练参数）
optimizer = torch.optim.AdamW(classifier.parameters(), lr=learning_rate)

# 训练循环
classifier.train()
for step in range(total_steps):
    # 获取batch
    xb, yb = get_classifier_batch(train_encoded, train_labels, batch_size, block_size, device)

    # 前向传播
    logits, loss = classifier(xb, yb)

    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 定期评估
    if step % eval_interval == 0:
        classifier.eval()

        # 训练集评估
        train_losses = []
        train_accs = []
        for _ in range(eval_iters):
            X, Y = get_classifier_batch(train_encoded, train_labels, batch_size, block_size, device)
            with torch.no_grad():
                logits, loss = classifier(X, Y)
                preds = (torch.sigmoid(logits) > 0.5).long()
                acc = (preds == Y).float().mean()
            train_losses.append(loss.item())
            train_accs.append(acc.item())

        # 验证集评估
        val_losses = []
        val_accs = []
        for _ in range(eval_iters):
            X, Y = get_classifier_batch(val_encoded, val_labels, batch_size, block_size, device)
            with torch.no_grad():
                logits, loss = classifier(X, Y)
                preds = (torch.sigmoid(logits) > 0.5).long()
                acc = (preds == Y).float().mean()
            val_losses.append(loss.item())
            val_accs.append(acc.item())

        print(f"Step {step:3d} | "
              f"训练 Loss: {sum(train_losses) / len(train_losses):.4f} Acc: {sum(train_accs) / len(train_accs):.2%} | "
              f"验证 Loss: {sum(val_losses) / len(val_losses):.4f} Acc: {sum(val_accs) / len(val_accs):.2%}")

        classifier.train()

print("\n✓ 训练完成")

Step   0 | 训练 Loss: 0.6874 Acc: 55.00% | 验证 Loss: 0.7061 Acc: 42.50%
Step  20 | 训练 Loss: 0.6971 Acc: 52.50% | 验证 Loss: 0.7017 Acc: 40.00%
Step  40 | 训练 Loss: 0.6902 Acc: 53.75% | 验证 Loss: 0.6962 Acc: 51.25%
Step  60 | 训练 Loss: 0.6945 Acc: 55.00% | 验证 Loss: 0.7157 Acc: 50.00%
Step  80 | 训练 Loss: 0.6903 Acc: 50.00% | 验证 Loss: 0.6946 Acc: 46.25%

✓ 训练完成


In [9]:
from pathlib import Path

# 保存分类器模型
output_path = Path("./output/05_author_classifier.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)

# 保存分类器的完整状态
torch.save({
    'classifier_state_dict': classifier.state_dict(),
    'vocab_size': tokenizer.vocab_size,
    'emb_size': 256,
    'block_size': 256,
    'layer_num': 8,
    'head_num': 8,
}, output_path)

print(f"分类器已保存到 {output_path}")

# 测试推理
classifier.eval()

# 构造测试样本（使用 encode_poem 格式）
test_poems = [
    {"author": "李白", "content": "床前明月光，疑是地上霜。举头望明月，低头思故乡。"},  # 正确配对
    {"author": "杜甫", "content": "床前明月光，疑是地上霜。举头望明月，低头思故乡。"},  # 错误配对（静夜思是李白的）
    {"author": "陆游", "content": "死去元知萬事空，但悲不見九州同。王師北定中原日，家祭無忘告乃翁。"},  # 正确配对
    {"author": "李白", "content": "死去元知萬事空，但悲不見九州同。王師北定中原日，家祭無忘告乃翁。"},  # 错误配对（示儿是陆游的）
]

print("\n测试推理：")
for poem_dict in test_poems:
    sample = encode_poem(poem_dict)
    test_ids = torch.tensor([tokenizer.encode(sample)], device=device)
    with torch.no_grad():
        logits = classifier(test_ids)
        prob = torch.sigmoid(logits)
    
    is_correct = (poem_dict["author"] == "李白" and "床前明月光" in poem_dict["content"]) or \
                 (poem_dict["author"] == "陆游" and "死去元知萬事空" in poem_dict["content"])
    label = "✓ 正确配对" if is_correct else "✗ 错误配对"
    
    print(f"  作者: {poem_dict['author']:4s} | 内容: {poem_dict['content'][:20]:20s}... | 预测概率: {prob.item():.2%} | {label}")

分类器已保存到 output/05_author_classifier.pt

测试推理：
  作者: 李白   | 内容: 床前明月光，疑是地上霜。举头望明月，低头... | 预测概率: 54.22% | ✓ 正确配对
  作者: 杜甫   | 内容: 床前明月光，疑是地上霜。举头望明月，低头... | 预测概率: 56.14% | ✗ 错误配对
  作者: 陆游   | 内容: 死去元知萬事空，但悲不見九州同。王師北定... | 预测概率: 52.64% | ✓ 正确配对
  作者: 李白   | 内容: 死去元知萬事空，但悲不見九州同。王師北定... | 预测概率: 51.89% | ✗ 错误配对


In [10]:
# ==================== 综合评价函数 ====================
from nanopoet.common import decode_poem_str

def compute_reward(
    poem_str: str,
    classifier: BinaryClassifier,
    tokenizer,
    format_data: dict,
    device: str,
    weights: dict = None
) -> dict:
    """
    综合评价函数，整合三个奖励目标
    
    Args:
        poem_str: 编码后的诗词字符串（如 "BA李白aC床前明月光...c"）
        classifier: 作者风格匹配分类器
        tokenizer: 分词器
        format_data: 格式数据字典
        device: 设备
        weights: 各项权重，默认 {'rhyme': 0.3, 'format': 0.3, 'author_style': 0.4}
    
    Returns:
        包含各项分数和总分的字典
    """
    # 默认权重
    if weights is None:
        weights = {
            'rhyme': 0.3,         # 押韵权重
            'format': 0.3,        # 格式权重
            'author_style': 0.4   # 作者风格匹配权重
        }
    
    # 解码诗词字符串
    poem_dict = decode_poem_str(poem_str)
    
    # 初始化结果
    result = {
        'rhyme_score': 0.0,
        'format_score': 0.0,
        'author_style_score': 0.0,
        'total_reward': 0.0,
        'details': {}
    }
    
    # 1. 押韵检查
    if 'content' in poem_dict:
        rhyme_score = check_rhyme(poem_dict['content'])
        result['rhyme_score'] = rhyme_score
        result['details']['rhyme'] = f"{rhyme_score:.2%}"
    
    # 2. 格式检查
    if 'style' in poem_dict and 'content' in poem_dict:
        format_score = check_format(poem_dict['style'], poem_dict['content'], format_data)
        result['format_score'] = float(format_score)
        result['details']['format'] = "✓ 通过" if format_score == 1 else "✗ 不通过"
    else:
        # 如果没有 style 字段，格式检查得满分（不做限制）
        result['format_score'] = 1.0
        result['details']['format'] = "⊘ 无格式要求"
    
    # 3. 作者风格匹配
    if 'author' in poem_dict and 'content' in poem_dict:
        classifier.eval()
        test_ids = torch.tensor([tokenizer.encode(poem_str)], device=device)
        with torch.no_grad():
            logits = classifier(test_ids)
            prob = torch.sigmoid(logits).item()
        result['author_style_score'] = prob
        result['details']['author_style'] = f"{prob:.2%}"
    else:
        # 如果没有 author 字段，作者匹配得满分（不做限制）
        result['author_style_score'] = 1.0
        result['details']['author_style'] = "⊘ 无作者要求"
    
    # 4. 计算总奖励（加权平均）
    total_reward = (
        weights['rhyme'] * result['rhyme_score'] +
        weights['format'] * result['format_score'] +
        weights['author_style'] * result['author_style_score']
    )
    result['total_reward'] = total_reward
    
    return result


def print_reward_analysis(poem_str: str, reward_result: dict):
    """打印奖励分析结果"""
    poem_dict = decode_poem_str(poem_str)
    
    print("=" * 70)
    print("诗词评价分析")
    print("=" * 70)
    
    # 诗词信息
    print("\n【诗词信息】")
    if 'author' in poem_dict:
        print(f"  作者: {poem_dict['author']}")
    if 'style' in poem_dict:
        print(f"  形式: {poem_dict['style']}")
    if 'title' in poem_dict:
        print(f"  标题: {poem_dict['title']}")
    if 'content' in poem_dict:
        print(f"  内容: {poem_dict['content']}")
    
    # 评分详情
    print("\n【评分详情】")
    print(f"  押韵检查:     {reward_result['details'].get('rhyme', 'N/A'):>10s} (权重: 30%)")
    print(f"  格式检查:     {reward_result['details'].get('format', 'N/A'):>10s} (权重: 30%)")
    print(f"  作者风格匹配: {reward_result['details'].get('author_style', 'N/A'):>10s} (权重: 40%)")
    
    # 总分
    print("\n【综合评分】")
    print(f"  总奖励: {reward_result['total_reward']:.2%}")
    
    # 评级
    score = reward_result['total_reward']
    if score >= 0.9:
        grade = "优秀 ★★★★★"
    elif score >= 0.8:
        grade = "良好 ★★★★"
    elif score >= 0.7:
        grade = "中等 ★★★"
    elif score >= 0.6:
        grade = "及格 ★★"
    else:
        grade = "待改进 ★"
    print(f"  评级: {grade}")
    print("=" * 70)


# ==================== 测试综合评价函数 ====================

# 测试1: 李白 - 静夜思（应该得高分）
test_case_1 = "BA李白aS五言绝句sT静夜思tC床前明月光，疑是地上霜。举头望明月，低头思故乡。c"
reward_1 = compute_reward(test_case_1, classifier, tokenizer, format_data, device)
print_reward_analysis(test_case_1, reward_1)

print("\n\n")

# 测试2: 错误作者（杜甫）- 静夜思（作者匹配分应该低）
test_case_2 = "BA杜甫aS五言绝句sT静夜思tC床前明月光，疑是地上霜。举头望明月，低头思故乡。c"
reward_2 = compute_reward(test_case_2, classifier, tokenizer, format_data, device)
print_reward_analysis(test_case_2, reward_2)

print("\n\n")

# 测试3: 只有内容，无作者和格式要求（应该只看押韵）
test_case_3 = "BC春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。c"
reward_3 = compute_reward(test_case_3, classifier, tokenizer, format_data, device)
print_reward_analysis(test_case_3, reward_3)

print("\n\n")

# 测试4: 格式错误（七言律诗但字数不对）
test_case_4 = "BA陆游aS七言律诗sC春风吹，春雨飘，春意闹。c"
reward_4 = compute_reward(test_case_4, classifier, tokenizer, format_data, device)
print_reward_analysis(test_case_4, reward_4)

诗词评价分析

【诗词信息】
  作者: 李白
  形式: 五言绝句
  标题: 静夜思
  内容: 床前明月光，疑是地上霜。举头望明月，低头思故乡。

【评分详情】
  押韵检查:        100.00% (权重: 30%)
  格式检查:           ✓ 通过 (权重: 30%)
  作者风格匹配:     52.47% (权重: 40%)

【综合评分】
  总奖励: 80.99%
  评级: 良好 ★★★★



诗词评价分析

【诗词信息】
  作者: 杜甫
  形式: 五言绝句
  标题: 静夜思
  内容: 床前明月光，疑是地上霜。举头望明月，低头思故乡。

【评分详情】
  押韵检查:        100.00% (权重: 30%)
  格式检查:           ✓ 通过 (权重: 30%)
  作者风格匹配:     53.61% (权重: 40%)

【综合评分】
  总奖励: 81.44%
  评级: 良好 ★★★★



诗词评价分析

【诗词信息】
  内容: 春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。

【评分详情】
  押韵检查:        100.00% (权重: 30%)
  格式检查:        ⊘ 无格式要求 (权重: 30%)
  作者风格匹配:    ⊘ 无作者要求 (权重: 40%)

【综合评分】
  总奖励: 100.00%
  评级: 优秀 ★★★★★



诗词评价分析

【诗词信息】
  作者: 陆游
  形式: 七言律诗
  内容: 春风吹，春雨飘，春意闹。

【评分详情】
  押韵检查:          0.00% (权重: 30%)
  格式检查:          ✗ 不通过 (权重: 30%)
  作者风格匹配:     52.96% (权重: 40%)

【综合评分】
  总奖励: 21.19%
  评级: 待改进 ★
